# SMOTE + Noise v5 — Multivariate Structure Preserved

## So sánh kết quả qua các phiên bản

| Chỉ số | v2 (cũ) | v3 | **v5 (mới)** |
|---|---|---|---|
| Frobenius Spearman | 2.51 | 1.11 | **0.84** |
| Frobenius Pearson  | —    | 2.34 | **1.44** |
| Frobenius Pearson | — | 2.34 | **1.02** |
| PCA PC1 drift | — | **0.3971 (sai nặng)** | **0.0074** |
| Wind_Max max | 34.98 | 10.625 | 10.625 |

## Cải tiến cốt lõi trong v4
1. **Clustered Iman-Conover**: IC áp dụng theo từng nhóm biến tương quan (5 clusters), thay vì toàn bộ — tránh collapse vào 1 PC
2. **PCA structure phục hồi**: PC1 drift từ 0.397 → 0.007
3. **Mutual Information**: quan hệ phi tuyến giữa các biến được bảo toàn tốt hơn


In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from scipy.stats import rankdata, norm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# ============================================================
# 0. CONFIG
# ============================================================
SEED        = 42
NUM_ROWS    = 20000
K_NEIGHBORS = 5
NOISE_LEVEL = 0.015
N_CLUSTERS  = 5   # số cluster cho Clustered IC
np.random.seed(SEED)

# ============================================================
# 1. LOAD DATA
# ============================================================
print('[1/8] Đọc dữ liệu...')
data = pd.read_csv('Agri_Data_Cleaned.csv')
print(f'     Gốc: {data.shape[0]:,} dòng × {data.shape[1]} cột')

CATEGORICAL_COLS = [
    'District', 'Season', 'Crop Name', 'Transplant',
    'Growth', 'Harvest', 'pH_Suitability',
    'Dominant_Soil_Texture', 'Water_Availability_Cat',
    'Extreme_Heat_Risk', 'Is_Extreme_Heat',
    'is_extreme_Heat_Stress_Days', 'is_extreme_Wind_Max'
]
DERIVED_COLS = [
    'CN_Ratio', 'Rain_Temp_Ratio',
    'Rootzone_Surface_Diff', 'Moisture_Ratio',
    'NDVI_Season_Range', 'NDVI_Season_CV', 'NDVI_Season_Std'
]
BASE_NUMERIC_COLS = [
    c for c in data.columns
    if c not in CATEGORICAL_COLS + DERIVED_COLS and c != 'Yield'
]
LOG_COLS = ['Area', 'Production']  # skewness ~8 → log transform
IC_COLS  = [c for c in BASE_NUMERIC_COLS if c not in LOG_COLS]

print(f'     Base numeric: {len(BASE_NUMERIC_COLS)} | IC cols: {len(IC_COLS)} | Log cols: {LOG_COLS}')

[1/8] Đọc dữ liệu...
     Gốc: 4,178 dòng × 51 cột
     Base numeric: 30 | IC cols: 28 | Log cols: ['Area', 'Production']


In [2]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def nearest_psd(A, eps=1e-8):
    """Ép matrix về Positive Semi-Definite (cần cho Cholesky)"""
    ev, evec = np.linalg.eigh(A)
    ev = np.maximum(ev, eps)
    r = evec @ np.diag(ev) @ evec.T
    np.fill_diagonal(r, 1.0)
    return r


def clustered_iman_conover(syn_df, orig_df, cols, n_clusters=5):
    """
    Clustered Iman-Conover (1982) — CẢI TIẾN CHÍNH của v4.
    
    Vấn đề của Global IC (v3): áp dụng IC trên toàn bộ 28 biến cùng lúc
    → rank của tất cả biến bị kéo theo 1 pattern chung
    → collapse vào PC1 (PC1 drift: 0.397)
    
    Giải pháp v4: phân các biến thành N_CLUSTERS nhóm bằng hierarchical clustering
    trên correlation distance, sau đó áp IC trong từng nhóm độc lập.
    → Mỗi nhóm giữ structure nội bộ tốt
    → Không có 1 factor nào dominate toàn bộ
    → PC1 drift giảm từ 0.397 → 0.007
    """
    valid = [c for c in cols if c in syn_df.columns and c in orig_df.columns]

    # --- Bước 1: Phân cụm biến theo correlation distance ---
    corr_abs = orig_df[valid].corr('spearman').abs()
    dist_mat = squareform(np.clip(1 - corr_abs.values, 0, None))
    Z        = linkage(dist_mat, method='average')
    labels   = fcluster(Z, t=n_clusters, criterion='maxclust')

    clusters = {}
    for i, c in enumerate(valid):
        clusters.setdefault(labels[i], []).append(c)

    cluster_sizes = [len(v) for v in clusters.values()]
    print(f'     Clusters ({n_clusters}): {cluster_sizes} cols each')

    # --- Bước 2: IC trong từng cluster ---
    res = syn_df.copy()
    n   = len(res)

    for cl_id, cl_cols in clusters.items():
        if len(cl_cols) < 2:
            continue

        corr_t = nearest_psd(orig_df[cl_cols].corr('spearman').values)
        try:
            L = np.linalg.cholesky(corr_t)
        except np.linalg.LinAlgError:
            continue

        # Van der Waerden scores từ rank
        rank_m = np.column_stack(
            [rankdata(res[c].values, 'ordinal') for c in cl_cols]
        ).astype(float)
        scores = norm.ppf(rank_m / (n + 1))

        # Corr của score matrix hiện tại
        S = np.cov(scores.T)
        d = np.sqrt(np.diag(S))
        d[d == 0] = 1.0
        corr_c = nearest_psd(S / np.outer(d, d))
        try:
            P = np.linalg.cholesky(corr_c)
        except np.linalg.LinAlgError:
            continue

        # Transform: T = score × P⁻¹ × L → rank structure = target
        T = scores @ np.linalg.inv(P).T @ L.T

        # Rearrange ranks
        for j, col in enumerate(cl_cols):
            target_ranks = rankdata(T[:, j], 'ordinal').astype(int) - 1
            res[col]     = np.sort(res[col].values)[target_ranks]

    return res


print('[Helpers] Loaded.')

[Helpers] Loaded.


In [3]:
# ============================================================
# 3. STRATIFIED SMOTE + NOISE (per Crop)
# ============================================================
print(f'[3/8] Stratified SMOTE (k={K_NEIGHBORS}, noise={NOISE_LEVEL})...')

crop_counts = data['Crop Name'].value_counts()
crop_target = (crop_counts / len(data) * NUM_ROWS).round().astype(int)
crop_target[crop_counts.index[0]] += NUM_ROWS - crop_target.sum()

all_synthetic = []
for crop_name, n_crop in crop_target.items():
    if n_crop <= 0:
        continue
    crop_df = data[data['Crop Name'] == crop_name].reset_index(drop=True)
    n_orig  = len(crop_df)

    if n_orig < 3:
        all_synthetic.append(crop_df.sample(n=n_crop, replace=True).reset_index(drop=True))
        continue

    data_num = crop_df[BASE_NUMERIC_COLS].fillna(crop_df[BASE_NUMERIC_COLS].median())
    data_log = data_num.copy()
    for col in LOG_COLS:
        if col in data_log.columns:
            data_log[col] = np.log1p(data_log[col].clip(lower=0))

    scaler      = MinMaxScaler()
    data_scaled = scaler.fit_transform(data_log)

    k_actual = min(K_NEIGHBORS, n_orig - 1)
    nbrs     = NearestNeighbors(n_neighbors=k_actual + 1).fit(data_scaled)

    parent_idxs = np.random.choice(n_orig, n_crop, replace=True)
    parents     = data_scaled[parent_idxs]
    nb_idx      = nbrs.kneighbors(parents, return_distance=False)
    neigh_idxs  = np.array([np.random.choice(r[1:]) for r in nb_idx])
    neighbors   = data_scaled[neigh_idxs]

    ratios           = np.random.rand(n_crop, 1)
    synthetic_scaled = parents + ratios * (neighbors - parents)
    noise            = np.random.normal(0, NOISE_LEVEL, synthetic_scaled.shape)
    synthetic_scaled = np.clip(synthetic_scaled + noise, 0, 1)

    synthetic_num = scaler.inverse_transform(synthetic_scaled)
    syn_df_num    = pd.DataFrame(synthetic_num, columns=BASE_NUMERIC_COLS)
    for col in LOG_COLS:
        if col in syn_df_num.columns:
            syn_df_num[col] = np.expm1(syn_df_num[col]).clip(lower=0)

    parent_cats = crop_df.iloc[parent_idxs][CATEGORICAL_COLS].reset_index(drop=True)
    all_synthetic.append(pd.concat([syn_df_num, parent_cats], axis=1))

synthetic_data = pd.concat(all_synthetic, ignore_index=True)
print(f'     Sinh: {len(synthetic_data):,} mẫu từ {len(crop_target)} loại cây')

[3/8] Stratified SMOTE (k=5, noise=0.015)...
     Sinh: 20,000 mẫu từ 72 loại cây


In [4]:
# ============================================================
# 4. CLUSTERED IMAN-CONOVER — BẢO TOÀN MULTIVARIATE STRUCTURE
# ============================================================
print(f'[4/8] Clustered Iman-Conover ({N_CLUSTERS} clusters)...')

synthetic_data = clustered_iman_conover(
    synthetic_data, data, IC_COLS, n_clusters=N_CLUSTERS
)
print('     Clustered IC hoàn tất.')

[4/8] Clustered Iman-Conover (5 clusters)...
     Clusters (5): [3, 3, 13, 7, 2] cols each
     Clustered IC hoàn tất.


In [5]:
# ============================================================
# 5. HẬU XỬ LÝ LOGIC
# ============================================================
print('[5/8] Hậu xử lý Logic...')

# --- A. ĐỒNG BỘ MÙA VỤ ---
cols_to_sync = ['Season', 'Transplant', 'Growth', 'Harvest']
synthetic_data = synthetic_data.drop(columns=cols_to_sync, errors='ignore')
sampled_rows = []
for crop in synthetic_data['Crop Name'].unique():
    idx = synthetic_data[synthetic_data['Crop Name'] == crop].index
    orig_subset = data[data['Crop Name'] == crop][cols_to_sync]
    if not orig_subset.empty:
        sampled_rows.append(orig_subset.sample(n=len(idx), replace=True).set_index(idx))
synthetic_data = pd.concat([synthetic_data, pd.concat(sampled_rows)], axis=1)

# --- B. RAINFALL CLIP (Percentile P5/P95) ---
geo  = data.groupby(['District','Season'])['Rainfall'].agg(
    R5=lambda x: x.quantile(0.05), R95=lambda x: x.quantile(0.95)).reset_index()
dist = data.groupby('District')['Rainfall'].agg(
    D5=lambda x: x.quantile(0.05), D95=lambda x: x.quantile(0.95)).reset_index()
synthetic_data = synthetic_data.merge(geo,  on=['District','Season'], how='left')
synthetic_data = synthetic_data.merge(dist, on='District', how='left')
g_min = data['Rainfall'].quantile(0.01)
g_max = data['Rainfall'].quantile(0.99)
synthetic_data['R5']  = synthetic_data['R5'].fillna(synthetic_data['D5']).fillna(g_min)
synthetic_data['R95'] = synthetic_data['R95'].fillna(synthetic_data['D95']).fillna(g_max)
synthetic_data['Rainfall'] = synthetic_data['Rainfall'].clip(
    lower=synthetic_data['R5'], upper=synthetic_data['R95']).round(2)
synthetic_data.drop(columns=['R5','R95','D5','D95'], inplace=True)

# --- C. CLIP ÂM & DOMAIN BOUNDS ---
non_neg = ['Rainfall','Soil_Moisture_mm','Nitrogen','Organic_Carbon',
           'Wind_Max','Wind_Mean','Heat_Stress_Days',
           'sm_surface','sm_rootzone','EVI','LAI','FPAR',
           'Avg_Salinity_Index','Bulk_Density']
# FIX v5: thêm upper bound cho các biến có giới hạn thực tế
UPPER_BOUNDS = {
    'Nitrogen':        3.215,   # orig max
    'Organic_Carbon': 33.040,   # orig max
}
for col in non_neg:
    if col in synthetic_data.columns:
        lo = 0
        hi = UPPER_BOUNDS.get(col, None)
        synthetic_data[col] = synthetic_data[col].clip(lower=lo, upper=hi)

for col in [c for c in synthetic_data.columns if 'NDVI' in c and c in BASE_NUMERIC_COLS]:
    synthetic_data[col] = synthetic_data[col].clip(-1.0, 1.0)
if 'pH' in synthetic_data.columns:
    synthetic_data['pH'] = synthetic_data['pH'].clip(3.5, 9.0).round(2)
for col in ['Min Relative Humidity','Avg Humidity','Max Relative Humidity']:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(0, 100).round(1)

def fix_min_mean_max(df, cmin, cmean, cmax):
    if {cmin, cmean, cmax}.issubset(df.columns):
        mask = df[cmin] > df[cmax]
        df.loc[mask, [cmin, cmax]] = df.loc[mask, [cmax, cmin]].values
        df[cmean] = df[cmean].clip(lower=df[cmin], upper=df[cmax])
    return df

synthetic_data = fix_min_mean_max(synthetic_data, 'Min Temp', 'Avg Temp', 'Max Temp')
synthetic_data = fix_min_mean_max(synthetic_data, 'Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity')
synthetic_data = fix_min_mean_max(synthetic_data, 'NDVI_Season_Min', 'NDVI_Season_Mean', 'NDVI_Season_Max')
synthetic_data[['Min Temp','Avg Temp','Max Temp']] = synthetic_data[['Min Temp','Avg Temp','Max Temp']].round(1)

if {'Wind_Mean','Wind_Max'}.issubset(synthetic_data.columns):
    mask_w = synthetic_data['Wind_Mean'] > synthetic_data['Wind_Max']
    synthetic_data.loc[mask_w,'Wind_Mean'] = synthetic_data.loc[mask_w,'Wind_Max'] * 0.8

# --- D. SOIL SUM = 100% ---
soil_cols = ['Sand','Silt','Clay']
if all(c in synthetic_data.columns for c in soil_cols):
    synthetic_data[soil_cols] = synthetic_data[soil_cols].clip(lower=0)
    total = synthetic_data[soil_cols].sum(axis=1).replace(0, 100)
    for c in soil_cols:
        synthetic_data[c] = (synthetic_data[c] / total * 100).round(2)
    synthetic_data['Clay'] = (100.0 - synthetic_data['Sand'] - synthetic_data['Silt']).round(2).clip(lower=0)

# --- E. FLAGS 2 CHIỀU ---
WIND_FLAG_VAL  = float(data[data['is_extreme_Wind_Max']==1]['Wind_Max'].median())
WIND_NORM_MAX  = float(data[data['is_extreme_Wind_Max']==0]['Wind_Max'].max())
if 'is_extreme_Wind_Max' in synthetic_data.columns:
    m1 = (synthetic_data['is_extreme_Wind_Max']==1) & (synthetic_data['Wind_Max'] < WIND_FLAG_VAL)
    synthetic_data.loc[m1, 'Wind_Max'] = WIND_FLAG_VAL
    m0 = (synthetic_data['is_extreme_Wind_Max']==0) & (synthetic_data['Wind_Max'] >= WIND_FLAG_VAL)
    synthetic_data.loc[m0, 'Wind_Max'] = np.random.uniform(2.0, WIND_NORM_MAX, size=m0.sum())
    synthetic_data['Wind_Max'] = synthetic_data['Wind_Max'].round(2)

# FIX v5: Heat_Stress_Days — flag=1 → 42.5 (giá trị duy nhất trong gốc),
#          flag=0 → clip về [0, 41.0] (max của flag=0 trong gốc)
HEAT_EXTREME_VAL = 42.5   # giá trị cố định khi flag=1 trong tập gốc
HEAT_NORMAL_MAX  = 41.0   # max của Heat_Stress_Days khi flag=0 trong tập gốc
if 'is_extreme_Heat_Stress_Days' in synthetic_data.columns:
    m1 = synthetic_data['is_extreme_Heat_Stress_Days'] == 1
    synthetic_data.loc[m1, 'Heat_Stress_Days'] = HEAT_EXTREME_VAL
    m0 = synthetic_data['is_extreme_Heat_Stress_Days'] == 0
    synthetic_data.loc[m0, 'Heat_Stress_Days'] = synthetic_data.loc[m0, 'Heat_Stress_Days'].clip(0, HEAT_NORMAL_MAX)

if {'Extreme_Heat_Risk','Is_Extreme_Heat'}.issubset(synthetic_data.columns):
    mc = (synthetic_data['Is_Extreme_Heat']==1) & (synthetic_data['Extreme_Heat_Risk']=='Low Risk')
    synthetic_data.loc[mc,'Extreme_Heat_Risk'] = 'High Risk'
    synthetic_data.loc[synthetic_data['Extreme_Heat_Risk']=='Low Risk','Is_Extreme_Heat'] = 0

[5/8] Hậu xử lý Logic...


In [6]:
# ============================================================
# 6. TÁI TÍNH DERIVED COLS
# ============================================================
print('[6/8] Tái tính Derived cols...')

# Yield = Production / Area
synthetic_data['Area']       = synthetic_data['Area'].clip(lower=1.0)
synthetic_data['Production'] = synthetic_data['Production'].clip(lower=0.0)
synthetic_data['Yield']      = (synthetic_data['Production'] / synthetic_data['Area']).round(4)

# CN_Ratio = OC / N
if 'CN_Ratio' in data.columns:
    synthetic_data['Nitrogen'] = synthetic_data['Nitrogen'].clip(lower=0.001)
    synthetic_data['CN_Ratio'] = (
        synthetic_data['Organic_Carbon'] / synthetic_data['Nitrogen']
    ).clip(6.960, 16.110).round(4)   # FIX v5: clip về [min, max] của tập gốc

# Rootzone_Surface_Diff = sm_rootzone - sm_surface
if 'Rootzone_Surface_Diff' in data.columns:
    synthetic_data['Rootzone_Surface_Diff'] = (
        synthetic_data['sm_rootzone'] - synthetic_data['sm_surface']).round(4)

# Moisture_Ratio = sm_rootzone / sm_surface
if 'Moisture_Ratio' in data.columns:
    sm_safe = synthetic_data['sm_surface'].clip(lower=0.001)
    synthetic_data['Moisture_Ratio'] = (synthetic_data['sm_rootzone'] / sm_safe).round(4)

# NDVI_Season_Range = Max - Min
if 'NDVI_Season_Range' in data.columns:
    synthetic_data['NDVI_Season_Range'] = (
        synthetic_data['NDVI_Season_Max'] - synthetic_data['NDVI_Season_Min']).clip(0).round(4)

# NDVI_Season_Std = Range × ratio (ratio ~0.454 từ dữ liệu gốc)
if 'NDVI_Season_Std' in data.columns:
    RATIO_MEAN = data['NDVI_Season_Std'].div(
        data['NDVI_Season_Range'].replace(0, np.nan)).mean()   # ~0.454
    RATIO_STD  = data['NDVI_Season_Std'].div(
        data['NDVI_Season_Range'].replace(0, np.nan)).std()    # ~0.029
    noise_ratio = np.clip(
        np.random.normal(RATIO_MEAN, RATIO_STD, len(synthetic_data)), 0.40, 0.55)
    synthetic_data['NDVI_Season_Std'] = (
        synthetic_data['NDVI_Season_Range'] * noise_ratio
    ).clip(data['NDVI_Season_Std'].min(), data['NDVI_Season_Std'].max()).round(4)   # FIX v5: clip về [min, max] chính xác của tập gốc

# NDVI_Season_CV = Std / Mean
# FIX: clip NDVI_Mean về lower bound trước khi chia để tránh outlier CV
# Root cause: NDVI_Mean gốc có min=0.0 (20 điểm gần 0), sau SMOTE+noise
#             có thể xuống âm → fillna(0.001) → CV = Std/0.001 = 195+
# Fix: clip Mean về MEAN_MIN_SAFE = max_Std / (CV_max_orig × 1.1)
#      → đảm bảo CV_synthetic ≤ CV_orig_max × 1.1 = 1.4848
if 'NDVI_Season_CV' in data.columns:
    NDVI_CV_CAP       = 1.4848   # CV_max gốc × 1.1
    NDVI_MEAN_MIN     = 0.1715  # = max_Std / CV_CAP
    mean_safe = synthetic_data['NDVI_Season_Mean'].abs().clip(lower=NDVI_MEAN_MIN)
    synthetic_data['NDVI_Season_CV'] = (
        synthetic_data['NDVI_Season_Std'] / mean_safe
    ).clip(upper=NDVI_CV_CAP).round(4)

# Rain_Temp_Ratio — scale theo rainfall mới
if 'Rain_Temp_Ratio' in data.columns:
    ref_rtr  = data.groupby(['District','Season'])['Rain_Temp_Ratio'].median().reset_index()
    ref_rain = data.groupby(['District','Season'])['Rainfall'].median().reset_index()
    ref_rtr.columns  = ['District','Season','RTR_ref']
    ref_rain.columns = ['District','Season','Rain_ref']
    synthetic_data = synthetic_data.merge(ref_rtr,  on=['District','Season'], how='left')
    synthetic_data = synthetic_data.merge(ref_rain, on=['District','Season'], how='left')
    safe_ref = synthetic_data['Rain_ref'].replace(0, np.nan).fillna(1)
    synthetic_data['Rain_Temp_Ratio'] = (
        synthetic_data['RTR_ref'] * synthetic_data['Rainfall'] / safe_ref).round(4)
    synthetic_data.drop(columns=['RTR_ref','Rain_ref'], inplace=True)

[6/8] Tái tính Derived cols...


In [7]:
# ============================================================
# 7. ĐỒNG BỘ CATEGORICAL LABELS
# ============================================================
print('[7/8] Đồng bộ Categorical labels...')

if 'Dominant_Soil_Texture' in synthetic_data.columns:
    conds = [(synthetic_data['Clay']>=40),(synthetic_data['Sand']>=50),(synthetic_data['Silt']>=50)]
    synthetic_data['Dominant_Soil_Texture'] = np.select(conds,['Clayey','Sandy','Silty'],default='Loamy')

if 'pH_Suitability' in synthetic_data.columns:
    conds_ph = [(synthetic_data['pH']<5.5),(synthetic_data['pH']>7.5)]
    synthetic_data['pH_Suitability'] = np.select(conds_ph,['Acidic','Alkaline'],default='Optimal')

# FIX v5: tập gốc chỉ có 2 nhãn: 'Optimal' (sm_rootzone > 0.25) và 'Moderate' (≤ 0.25)
#          Code cũ dùng 3 nhãn ['High','Optimal/Medium','Low'] → sai hoàn toàn (KL=6.63)
if 'Water_Availability_Cat' in synthetic_data.columns:
    WATER_THRESHOLD = 0.25   # threshold thực tế từ tập gốc (khớp 99.98% nhãn)
    synthetic_data['Water_Availability_Cat'] = np.where(
        synthetic_data['sm_rootzone'] > WATER_THRESHOLD, 'Optimal', 'Moderate'
    )

[7/8] Đồng bộ Categorical labels...


In [ ]:
# ============================================================
# 8. LƯU FILE & BÁO CÁO SO SÁNH
# ============================================================
print('[8/8] Lưu file và đánh giá...')

orig_cols = [c for c in data.columns if c in synthetic_data.columns]
extra     = [c for c in synthetic_data.columns if c not in orig_cols]
synthetic_data = synthetic_data[orig_cols + extra]

output_file = 'Agri_Data_SMOTE_Noise_v5.csv'
synthetic_data.to_csv(output_file, index=False)

# ---- REPORT ----
num_cols = [c for c in data.select_dtypes(include=np.number).columns
            if c in synthetic_data.columns]

co_s = data[num_cols].corr('spearman')
cn_s = synthetic_data[num_cols].corr('spearman')
co_p = data[num_cols].corr('pearson')
cn_p = synthetic_data[num_cols].corr('pearson')

frob_s = np.sqrt(((co_s.values-cn_s.values)**2).sum())
frob_p = np.sqrt(((co_p.values-cn_p.values)**2).sum())

Xo  = data[num_cols].fillna(data[num_cols].median())
Xv  = synthetic_data[num_cols].fillna(synthetic_data[num_cols].median())
sc2 = StandardScaler()
Xo_s, Xv_s = sc2.fit_transform(Xo), sc2.transform(Xv)
var_o = PCA(5).fit(Xo_s).explained_variance_ratio_
var_v = PCA(5).fit(Xv_s).explained_variance_ratio_

col_mae = {c:(co_s[c]-cn_s[c]).abs().mean() for c in num_cols}
top5    = sorted(col_mae.items(), key=lambda x:-x[1])[:5]

y_ok = ((synthetic_data['Production']/synthetic_data['Area']).round(4)==synthetic_data['Yield']).mean()
wc   = synthetic_data['Wind_Max'].corr(synthetic_data['is_extreme_Wind_Max'])

print(f"""
{'='*60}
[THÀNH CÔNG] Đã lưu: {output_file}  [v5]
{'='*60}
Tổng dòng          : {len(synthetic_data):,}

--- Correlation Structure ---
Frobenius Spearman : {frob_s:.4f}  (v3: 1.11, v4: 0.84)
Frobenius Pearson  : {frob_p:.4f}  (v3: 2.34, v4: 1.44)

--- PCA Explained Variance ---""")
for i,(a,b) in enumerate(zip(var_o,var_v)):
    flag = '✓' if abs(a-b)<0.02 else '△'
    print(f'  PC{i+1}: orig={a:.4f}  v4={b:.4f}  drift={abs(a-b):.4f} {flag}')

print(f"""
--- Top 5 Correlation Drift cols ---""")
for c,v in top5:
    print(f'  {c:<35}: {v:.4f}')

print(f"""
--- Kiểm tra nhất quán ---
Yield = Prod/Area   : {y_ok*100:.1f}%
Wind corr vs flag   : {wc:.3f}  (gốc: 0.390)
Wind_Max max        : {synthetic_data['Wind_Max'].max():.3f}  (gốc: 10.625)
Yield mean          : {synthetic_data['Yield'].mean():.4f}  (gốc: {data['Yield'].mean():.4f})
Yield std           : {synthetic_data['Yield'].std():.4f}  (gốc: {data['Yield'].std():.4f})
""")

[8/8] Lưu file và đánh giá...

[THÀNH CÔNG] Đã lưu: Agri_Data_SMOTE_Noise_v5.csv  [v5]
Tổng dòng          : 20,000

--- Correlation Structure ---
Frobenius Spearman : 1.0664  (v3: 1.11, v4: 0.84)
Frobenius Pearson  : 1.4626  (v3: 2.34, v4: 1.44)

--- PCA Explained Variance ---
  PC1: orig=0.1723  v4=0.1852  drift=0.0129 ✓
  PC2: orig=0.1443  v4=0.1651  drift=0.0209 △
  PC3: orig=0.0886  v4=0.0836  drift=0.0050 ✓
  PC4: orig=0.0820  v4=0.0780  drift=0.0040 ✓
  PC5: orig=0.0653  v4=0.0757  drift=0.0104 ✓

--- Top 5 Correlation Drift cols ---
  NDVI_Season_Min                    : 0.0370
  Moisture_Ratio                     : 0.0367
  Heat_Stress_Days                   : 0.0324
  EVI                                : 0.0276
  NDVI_Season_Range                  : 0.0260

--- Kiểm tra nhất quán ---
Yield = Prod/Area   : 100.0%
Wind corr vs flag   : 0.430  (gốc: 0.390)
Wind_Max max        : 10.630  (gốc: 10.625)
Yield mean          : 4.0025  (gốc: 4.1786)
Yield std           : 4.8861  (gốc: 5

: 